In [ ]:
import kaggle_benchmarks as kbench
import json
import re
import math
from datetime import datetime

# ----------------------------
# Global trace store
# ----------------------------
TRACE_LOG = []

FAILURE_MODES = [
    "failure_to_recognize_key_aspects",
    "hallucination",
    "misapplication_of_equation_or_model",
    "incorrect_factual_knowledge",
    "calculation_error",
]

# Fixed verifier substitution point
VERIFIER_SUBS = {
    "k": 5.0,
    "gamma": 0.2,
    "m": 2.0,
    "r0": 4.0,
}

CANONICAL_VALUE = VERIFIER_SUBS["gamma"] / (
    math.sqrt(VERIFIER_SUBS["m"] * VERIFIER_SUBS["k"]) * VERIFIER_SUBS["r0"] ** 3.5
)

REL_TOL = 1e-8
ABS_TOL = 1e-10

# ----------------------------
# Helpers
# ----------------------------
def extract_json(text):
    if not text:
        return None

    fence = re.search(r"```(?:json)?\s*(\{.*?\})\s*```", text, re.DOTALL)
    if fence:
        blob = fence.group(1)
    else:
        start = text.find("{")
        end = text.rfind("}")
        if start == -1 or end == -1 or end <= start:
            return None
        blob = text[start:end + 1]

    try:
        return json.loads(blob)
    except Exception:
        return None

def safe_get_attr(obj, attr_name, default=None):
    try:
        return getattr(obj, attr_name, default)
    except Exception:
        return default

def build_trace(
    *,
    task_id,
    llm,
    prompt,
    response,
    parsed,
    final_answer,
    normalized_answer,
    passed,
    failure_mode,
):
    return {
        "timestamp_utc": datetime.utcnow().isoformat() + "Z",
        "task_id": task_id,
        "model": str(llm),
        "pass": bool(passed),
        "failure_mode": failure_mode,
        "final_answer": final_answer,
        "normalized_answer": normalized_answer,
        "raw_output": response,
        "parsed_output": parsed,
        "prompt": prompt,
        "tokens_input": safe_get_attr(llm, "last_input_tokens"),
        "tokens_output": safe_get_attr(llm, "last_output_tokens"),
        "cost": safe_get_attr(llm, "last_cost"),
        "latency_ms": safe_get_attr(llm, "last_latency_ms"),
    }

# ----------------------------
# Symbolic answer normalization
# ----------------------------
def normalize_symbolic_text(text):
    if text is None:
        return ""

    s = str(text)

    replacements = {
        "$": "",
        "\\left": "",
        "\\right": "",
        "\\,": "",
        "\\;": "",
        "\\quad": ",",
        "\\qquad": ",",
        "−": "-",
        "–": "-",
        "—": "-",
        "γ": "gamma",
        "Ω": "Omega",
        "ω": "omega",
        "κ": "kappa",
        "\\gamma": "gamma",
        "\\Omega": "Omega",
        "\\omega": "omega",
        "\\kappa": "kappa",
        "\\sqrt": "sqrt",
        "\\cdot": "*",
        "\\approx": "=",
        "\\to": "=",
        "r_0": "r0",
        "r_{0}": "r0",
    }

    for old, new in replacements.items():
        s = s.replace(old, new)

    # remove \text{...}
    s = re.sub(r"\\text\{.*?\}", "", s)

    # convert \frac{A}{B} recursively
    while "\\frac" in s:
        m = re.search(r"\\frac\s*\{", s)
        if not m:
            break
        start = m.start()

        def extract_braced(src, idx):
            if idx < 0 or idx >= len(src) or src[idx] != "{":
                return None, None
            depth = 0
            for j in range(idx, len(src)):
                if src[j] == "{":
                    depth += 1
                elif src[j] == "}":
                    depth -= 1
                    if depth == 0:
                        return src[idx + 1:j], j
            return None, None

        num_start = s.find("{", start)
        num, num_end = extract_braced(s, num_start)
        if num is None:
            break
        den_start = s.find("{", num_end + 1)
        den, den_end = extract_braced(s, den_start)
        if den is None:
            break

        repl = f"(({num})/({den}))"
        s = s[:start] + repl + s[den_end + 1:]

    s = s.replace("{", "(").replace("}", ")")
    s = re.sub(r"\s+", "", s)

    return s

def split_symbolic_candidates(answer_text):
    s = normalize_symbolic_text(answer_text)
    if not s:
        return [], s

    # First cut off explanatory tails after commas / semicolons
    head = re.split(r"[,;]", s)[0]

    # Split chains like a=b=c
    raw_parts = [p for p in head.split("=") if p]

    candidates = []
    for part in raw_parts:
        # Drop obvious LHS labels
        if part.lower() in {"omega", "dotomega", "Omega", "kappa", "omega-kappa"}:
            continue
        candidates.append(part)

    # Also keep full head if useful
    if head and head not in candidates:
        candidates.append(head)

    return candidates, s

# ----------------------------
# Expression evaluation
# ----------------------------
def convert_to_python_expr(expr):
    e = expr

    # common product shorthands
    e = e.replace("mk", "m*k")
    e = e.replace("km", "k*m")

    # powers
    e = e.replace("^", "**")
    e = e.replace("r0**7/2", "r0**(7/2)")
    e = e.replace("r0^(7/2)", "r0**(7/2)")
    e = e.replace("r0**(7/2)", "r0**(7/2)")

    # sqrt(...)
    e = re.sub(r"sqrt\(([^()]+)\)", r"math.sqrt(\1)", e)

    # handle common exact equivalent form gamma/sqrt(k*m*r0^7)
    e = e.replace("math.sqrt(k*m*r0**7)", "math.sqrt(k*m*(r0**7))")
    e = e.replace("math.sqrt(m*k*r0**7)", "math.sqrt(m*k*(r0**7))")

    return e

def eval_candidate(expr, subs):
    pyexpr = convert_to_python_expr(expr)

    env = {
        "__builtins__": {},
        "math": math,
        "gamma": subs["gamma"],
        "k": subs["k"],
        "m": subs["m"],
        "r0": subs["r0"],
        "Omega": math.sqrt(subs["k"] / (subs["m"] * subs["r0"] ** 3)),
    }

    return eval(pyexpr, env, {})

def symbolic_verifier(answer_text):
    candidates, normalized = split_symbolic_candidates(answer_text)
    if not candidates:
        return False, "hallucination", normalized

    for cand in candidates:
        try:
            value = eval_candidate(cand, VERIFIER_SUBS)
            if math.isfinite(value) and math.isclose(
                value, CANONICAL_VALUE, rel_tol=REL_TOL, abs_tol=ABS_TOL
            ):
                return True, None, normalized
        except Exception:
            continue

    return False, None, normalized

def classify_failure_fp_0009(answer_text):
    s = normalize_symbolic_text(answer_text)
    if not s:
        return "hallucination"

    # classic wrong answer: precession angle per orbit, not temporal rate
    if "2*pi*gamma" in s and "k*r0^2" in s:
        return "failure_to_recognize_key_aspects"

    if "perorbit" in s or "radiansperorbit" in s:
        return "failure_to_recognize_key_aspects"

    if "gamma" in s:
        return "misapplication_of_equation_or_model"

    return "hallucination"

# ----------------------------
# Frontier Physics Task 009
# ----------------------------
@kbench.task(
    name="FP-0009 Relativistic Correction Periapsis Precession Rate",
    description="Easy classical-mechanics task distinguishing periapsis advance per orbit from the temporal precession rate, with verifier-side numeric substitution for symbolic answers."
)
def fp_0009_periapsis_precession_rate(llm) -> tuple[int, int]:
    prompt = r"""You are solving a physics problem. Return valid JSON only — no prose outside the JSON.

A space probe of mass $m$ is orbiting a dense neutron star. Due to strong-field relativistic effects, the gravitational field is not a perfect Newtonian inverse-square. Instead, the force experienced by the probe is modeled by:
$F(r) = - \frac{k}{r^2} - \frac{\gamma}{r^4}$.

Here, $k$ is the standard gravitational parameter $GMm$, and $\gamma$ is a small positive constant representing the relativistic correction, with $\gamma \ll k r^2$.

The probe settles into a nearly circular orbit of radius $r_0$.

Question: What is the rate of precession of the orbit's periapsis? Express your answer in terms of $k$, $\gamma$, $m$, and $r_0$.

Return JSON only in the following format:
{
  "final_answer": "<symbolic expression>"
}"""

    response = llm.prompt(prompt)
    parsed = extract_json(response)

    total_checks = 1
    passed_checks = 0
    final_answer = ""
    normalized_answer = ""
    failure_mode = None

    if parsed is None:
        failure_mode = "hallucination"
    else:
        final_answer = parsed.get("final_answer", "")
        correct, code_failure, normalized_answer = symbolic_verifier(final_answer)

        if correct:
            passed_checks = 1
        else:
            failure_mode = code_failure or classify_failure_fp_0009(final_answer)

    trace = build_trace(
        task_id="fp_0009",
        llm=llm,
        prompt=prompt,
        response=response,
        parsed=parsed,
        final_answer=final_answer,
        normalized_answer=normalized_answer,
        passed=(passed_checks == 1),
        failure_mode=failure_mode,
    )
    TRACE_LOG.append(trace)

    return (passed_checks, total_checks)

In [ ]:
fp_0009_periapsis_precession_rate.run(kbench.llm)

In [ ]:
results = fp_0009_periapsis_precession_rate.evaluate(llm=[kbench.llm])
results.as_dataframe()

In [ ]:
import pandas as pd

trace_df = pd.DataFrame(TRACE_LOG)
trace_df[trace_df["task_id"] == "fp_0009"]